In [1]:
from pathlib import Path

CUAD_ROOT = Path("/kaggle/input/datasets/riyasuryawanshi/cuad-project")
SCAN_ROOT = Path("/kaggle/input/datasets/riyasuryawanshi/scan-vs-span")

print("CUAD project:", CUAD_ROOT.exists())
print("Indian set:", (CUAD_ROOT / "indian_set").exists())
print("Scan-vs-span:", SCAN_ROOT.exists())

for name in [
    "train_phase3.py",
    "scan_vs_span.py",
    "model.py",
    "dataset.py",
    "metrics.py",
    "ckpt_joint_cls_span_fold0.pt",
    "partial/config.json",
]:
    print(f"{name}: {(SCAN_ROOT / name).exists()}")

print("Indian evaluator:",
      (CUAD_ROOT / "indian_set" / "eval_indian.py").exists())
print("Indian builder:",
      (CUAD_ROOT / "indian_set" / "build_indian_testset.py").exists())

CUAD project: True
Indian set: True
Scan-vs-span: True
train_phase3.py: True
scan_vs_span.py: True
model.py: True
dataset.py: True
metrics.py: True
ckpt_joint_cls_span_fold0.pt: True
partial/config.json: True
Indian evaluator: True
Indian builder: True


In [2]:
from pathlib import Path
import shutil
import os

WORK = Path("/kaggle/working/indian_eval_project")

if WORK.exists():
    shutil.rmtree(WORK)

shutil.copytree(SCAN_ROOT, WORK)

print("Working copy created:", WORK)
print("train_phase3.py:", (WORK / "train_phase3.py").exists())
print("scan_vs_span.py:", (WORK / "scan_vs_span.py").exists())
print("Checkpoint:", (WORK / "ckpt_joint_cls_span_fold0.pt").exists())
print("Config:", (WORK / "partial/config.json").exists())

Working copy created: /kaggle/working/indian_eval_project
train_phase3.py: True
scan_vs_span.py: True
Checkpoint: True
Config: True


In [3]:
from pathlib import Path
import shutil

src = CUAD_ROOT / "indian_set"
dst = WORK / "indian_set"

assert src.exists(), f"Indian set not found: {src}"

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print("Indian set copied:", dst)
print("Documents folder:", (dst / "documents").exists())
print("Builder:", (dst / "build_indian_testset.py").exists())
print("Evaluator:", (dst / "eval_indian.py").exists())

print("\nFiles copied:")
for p in sorted(dst.iterdir()):
    print(p.name)

Indian set copied: /kaggle/working/indian_eval_project/indian_set
Documents folder: False
Builder: True
Evaluator: True

Files copied:
aera_rfp_consultant_2025.txt
build_indian_testset.py
clause_browser.csv
documents_manifest.csv
dot_unified_licence_agreement.txt
eval_indian.py
gem_general_terms_and_conditions.txt
himachal_hemodialysis_ppp_concession_agreement.txt
mea_architect_consultancy_agreement.txt
mea_pmc_services_agreement.txt
meity_msa_cloud_services_template.txt
mha_rfp_project_management_consultants.txt
noida_airport_concession_agreement.txt
prepare_documents.py
smoke_test.py
stqc_non_disclosure_agreement.txt
tender_consultancy_scc_1349.txt


In [4]:
from pathlib import Path
import shutil

indian = WORK / "indian_set"
docs_dir = indian / "documents"
docs_dir.mkdir(exist_ok=True)

# Copy only agreement text files into documents/
txt_files = list(indian.glob("*.txt"))

for src in txt_files:
    shutil.copy2(src, docs_dir / src.name)

print("Documents copied:", len(list(docs_dir.glob("*.txt"))))
print("\nFiles:")
for p in sorted(docs_dir.glob("*.txt")):
    print(p.name, "-", p.stat().st_size, "bytes")

Documents copied: 11

Files:
aera_rfp_consultant_2025.txt - 117936 bytes
dot_unified_licence_agreement.txt - 357403 bytes
gem_general_terms_and_conditions.txt - 75017 bytes
himachal_hemodialysis_ppp_concession_agreement.txt - 201172 bytes
mea_architect_consultancy_agreement.txt - 24286 bytes
mea_pmc_services_agreement.txt - 23618 bytes
meity_msa_cloud_services_template.txt - 107511 bytes
mha_rfp_project_management_consultants.txt - 87392 bytes
noida_airport_concession_agreement.txt - 769972 bytes
stqc_non_disclosure_agreement.txt - 15055 bytes
tender_consultancy_scc_1349.txt - 9756 bytes


In [6]:
from pathlib import Path
import csv

indian = WORK / "indian_set"
clauses_file = indian / "clauses.csv"

with clauses_file.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "document_file",
        "category",
        "clause_text",
        "source_note"
    ])

print("Created:", clauses_file)
print("Columns:", next(csv.reader(
    clauses_file.open(encoding="utf-8")
)))

Created: /kaggle/working/indian_eval_project/indian_set/clauses.csv
Columns: ['document_file', 'category', 'clause_text', 'source_note']


In [7]:
from pathlib import Path

doc_path = WORK / "indian_set" / "documents" / "aera_rfp_consultant_2025.txt"
text = doc_path.read_text(encoding="utf-8", errors="replace")

print("Document:", doc_path.name)
print("Characters:", len(text))

for term in ["Confidentiality", "Indemnification"]:
    print(f"\n{term}:")
    start = text.lower().find(term.lower())
    if start == -1:
        print("Not found")
    else:
        print(text[max(0, start-100):start+500])

Document: aera_rfp_consultant_2025.txt
Characters: 117602

Confidentiality:
ification and post-qualification experience as requested in bid document.

3. I shall maintain full confidentiality with respect to the material, information, document, etc. as may be made available to me during the proposed assignment.

(Signature and name of the Key Personnel)

(Signature, name and designation) Duly authorized to sign for and on behalf of bidder Seal of the Bidder

Note:

1. Use separate form for each Key Personnel.

2. This form to be signed by respective Key Personnel and the Authorized Signatory of the Bidder.

Form – 4

Assignment Details of the Bidder

S. No.

Details o

Indemnification:
Not found


In [8]:
import re

doc_path = WORK / "indian_set" / "documents" / "aera_rfp_consultant_2025.txt"
text = doc_path.read_text(encoding="utf-8", errors="replace")

for term in ["confidential", "indemnif", "hold harmless"]:
    matches = list(re.finditer(term, text, re.IGNORECASE))
    print(f"\n{term!r}: {len(matches)} matches")

    for m in matches[:5]:
        start = max(0, m.start() - 150)
        end = min(len(text), m.end() + 350)
        print("-" * 50)
        print(text[start:end].replace("\n", " "))


'confidential': 13 matches
--------------------------------------------------
resented. Further, I have requisite essential qualification and post-qualification experience as requested in bid document.  3. I shall maintain full confidentiality with respect to the material, information, document, etc. as may be made available to me during the proposed assignment.  (Signature and name of the Key Personnel)  (Signature, name and designation) Duly authorized to sign for and on behalf of bidder Seal of the Bidder  Note:  1. Use separate form for each Key Personnel.  2. This form to be sig
--------------------------------------------------
pplicable Laws” means the laws and any other instruments having the force of law in India as they may be issued and in force from time to time;  d)  “Confidential Information” shall have the meaning set forth in Clause 3.3;  e)  “Conflict of Interest” shall have the meaning set forth in Clause 3.2 read with the provisions of  RFP;  f)  “Dispute” shall hav

In [9]:
from pathlib import Path
import csv
import re

indian = WORK / "indian_set"
doc_name = "aera_rfp_consultant_2025.txt"
doc_path = indian / "documents" / doc_name
text = doc_path.read_text(encoding="utf-8", errors="replace")

# Find the Confidentiality heading anywhere in the document
conf_match = re.search(r"Confidentiality\s+The Consultant and its Key Personnel", text, re.I)
if not conf_match:
    raise ValueError("Could not locate the Confidentiality clause.")

conf_start = conf_match.start()
conf_end_match = re.search(r"\n\s*(?:3\.\d+(?:\.\d+)?\s+|4\.\d+(?:\.\d+)?\s+)", text[conf_match.end():])
conf_end = conf_match.end() + conf_end_match.start() if conf_end_match else conf_match.end() + 5000
conf_clause = text[conf_start:conf_end].strip()

# Find Indemnification clause 3.6.4
ind_match = re.search(r"3\.6\.4\s+The Consultant shall hold the Authority", text, re.I)
if not ind_match:
    raise ValueError("Could not locate clause 3.6.4.")

ind_next = re.search(r"3\.6\.5\b", text[ind_match.end():])
ind_end = ind_match.end() + ind_next.start() if ind_next else min(len(text), ind_match.end() + 3000)
ind_clause = text[ind_match.start():ind_end].strip()

rows = [
    [doc_name, "Confidentiality", conf_clause, "AERA agreement; Confidentiality section"],
    [doc_name, "Indemnification", ind_clause, "AERA agreement; clause 3.6.4"],
]

out = indian / "clauses.csv"
with out.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["document_file", "category", "clause_text", "source_note"])
    writer.writerows(rows)

print("Saved:", out)
print("Annotations:", len(rows))
print("Confidentiality characters:", len(conf_clause))
print("Indemnification characters:", len(ind_clause))
print("Confidentiality exact source match:", conf_clause in text)
print("Indemnification exact source match:", ind_clause in text)

Saved: /kaggle/working/indian_eval_project/indian_set/clauses.csv
Annotations: 2
Confidentiality characters: 2523
Indemnification characters: 3046
Confidentiality exact source match: True
Indemnification exact source match: True


In [11]:
import subprocess
import sys
import os

builder = WORK / "indian_set" / "build_indian_testset.py"

env = os.environ.copy()
env["PYTHONPATH"] = (
    str(WORK) + os.pathsep +
    str(WORK / "indian_set") + os.pathsep +
    env.get("PYTHONPATH", "")
)

result = subprocess.run(
    [sys.executable, str(builder), "--help"],
    capture_output=True,
    text=True,
    cwd=str(WORK),
    env=env
)

print("Return code:", result.returncode)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

Return code: 0
usage: build_indian_testset.py [-h] [--docs_dir DOCS_DIR]
                               [--clauses_csv CLAUSES_CSV] [--out_dir OUT_DIR]
                               [--make_template DIR] [--list_categories]

build_indian_testset.py -- turn your hand-collected Indian clauses into a test set
the Phase 3 code can evaluate.

YOU PROVIDE
    indian_set/documents/*.txt   full plain text of each Indian agreement / template
    indian_set/clauses.csv       one row per clause you picked
        document_file   file name inside documents/ (e.g. nda_sample_1.txt)
        category        one of the 18 ClauseXplain categories (exact spelling, see --list_categories)
        clause_text     the clause, COPIED from that document (whitespace differences are fine)
        source_note     where the document came from (public template, tender, ...)

THIS SCRIPT
    finds each clause inside its document (so we get exact character offsets, which
    the span evaluation needs), and writes
 

In [12]:
import subprocess
import sys
import os

env = os.environ.copy()
env["PYTHONPATH"] = (
    str(WORK) + os.pathsep +
    str(WORK / "indian_set") + os.pathsep +
    env.get("PYTHONPATH", "")
)

result = subprocess.run(
    [
        sys.executable,
        str(WORK / "indian_set" / "build_indian_testset.py"),
        "--list_categories"
    ],
    capture_output=True,
    text=True,
    cwd=str(WORK),
    env=env
)

print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

Assignment
Commercial Terms
Dispute Resolution
Governing Law / Jurisdiction
Intellectual Property
Liability
Non-Compete
Non-Solicitation
Other
Penalties
Termination
Warranty
Confidentiality
Force Majeure
Indemnification
Modification
Payment
Privacy / Data Protection



In [13]:
import subprocess
import sys
import os

env = os.environ.copy()
env["PYTHONPATH"] = (
    str(WORK) + os.pathsep +
    str(WORK / "indian_set") + os.pathsep +
    env.get("PYTHONPATH", "")
)

result = subprocess.run(
    [
        sys.executable,
        str(WORK / "indian_set" / "build_indian_testset.py"),
        "--docs_dir", str(WORK / "indian_set" / "documents"),
        "--clauses_csv", str(WORK / "indian_set" / "clauses.csv"),
        "--out_dir", str(WORK / "indian_set" / "built")
    ],
    capture_output=True,
    text=True,
    cwd=str(WORK),
    env=env
)

print("Return code:", result.returncode)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

Return code: 0
# Indian test set build report

2 clauses from 1 documents.

| Category | clauses | model has training data? |
|---|---|---|
| Confidentiality | 1 | no (never seen in training) |
| Indemnification | 1 | no (never seen in training) |

Categories with fewer than 3 clauses (too thin for per-category scores): Confidentiality, Indemnification

CSV encoding read as: utf-8

## Clauses per document

- aera_rfp_consultant_2025.txt: 2

Clauses in never-seen categories: 2 of 2
NOTE: Dispute Resolution is only 'seen' via CUAD's Covenant Not To Sue (no arbitration/forum clauses), so score it separately from the other seen categories.

## Warnings (rows kept, but check them)
- line 3: text ends in dot leaders or a page number -- probably a table-of-contents line



In [14]:
from pathlib import Path
import json

built = WORK / "indian_set" / "built"

report = built / "build_report.md"
mapped = built / "indian_mapped.jsonl"
contracts = built / "indian_contracts.json"

print("Generated files:")
for p in [report, mapped, contracts]:
    print(p.name, "exists:", p.exists())

print("\n--- Build report ---")
print(report.read_text(encoding="utf-8"))

print("\n--- Mapped clauses ---")
for line in mapped.read_text(encoding="utf-8").splitlines():
    row = json.loads(line)
    print(json.dumps(row, indent=2, ensure_ascii=False)[:2500])

Generated files:
build_report.md exists: True
indian_mapped.jsonl exists: True
indian_contracts.json exists: True

--- Build report ---
# Indian test set build report

2 clauses from 1 documents.

| Category | clauses | model has training data? |
|---|---|---|
| Confidentiality | 1 | no (never seen in training) |
| Indemnification | 1 | no (never seen in training) |

Categories with fewer than 3 clauses (too thin for per-category scores): Confidentiality, Indemnification

CSV encoding read as: utf-8

## Clauses per document

- aera_rfp_consultant_2025.txt: 2

Clauses in never-seen categories: 2 of 2
NOTE: Dispute Resolution is only 'seen' via CUAD's Covenant Not To Sue (no arbitration/forum clauses), so score it separately from the other seen categories.

## Warnings (rows kept, but check them)
- line 3: text ends in dot leaders or a page number -- probably a table-of-contents line

--- Mapped clauses ---
{
  "clause_id": "indian_aera_rfp_consultant_2025__Confidentiality__0",
  "contra

In [15]:
import csv
from pathlib import Path

indian = WORK / "indian_set"
csv_path = indian / "clauses.csv"

with csv_path.open(encoding="utf-8", newline="") as f:
    rows = list(csv.DictReader(f))

for i, row in enumerate(rows, start=1):
    clause = row["clause_text"].strip()
    print(f"\nRow {i}: {row['category']}")
    print("Document:", row["document_file"])
    print("Characters:", len(clause))
    print("Starts:", repr(clause[:180]))
    print("Ends:", repr(clause[-250:]))
    print("Contains next section heading 3.7:", "3.7" in clause)
    print("Contains dot leaders:", bool(__import__("re").search(r"\.{3,}", clause)))


Row 1: Confidentiality
Document: aera_rfp_consultant_2025.txt
Characters: 2523
Starts: 'Confidentiality The Consultant and its Key Personnel shall not, either during the term or within one year after the expiration or termination of this Agreement disclose any proprie'
Ends: 'formation, confidential and shall use its best efforts to ensure compliance with such undertaking.\n\nSubject to the confidentiality obligations above, the Consultant shall be allowed to use this engagement as an experience citation with other clients.'
Contains next section heading 3.7: False
Contains dot leaders: False

Row 2: Indemnification
Document: aera_rfp_consultant_2025.txt
Characters: 3046
Starts: '3.6.4\n\nThe Consultant shall hold the Authority harmless and indemnified for any losses, claims, damages, expenses (including all legal expenses), awards, penalties or injuries (col'
Ends: 'Key Personnel as may be required to carry out the Services.\n\n4.2\n\nDeployment of Key Personnel A list of Key Personn

In [16]:
from pathlib import Path
import subprocess
import sys
import os

WORK = Path("/kaggle/working/indian_eval_project")
indian = WORK / "indian_set"

env = os.environ.copy()
env["PYTHONPATH"] = (
    str(WORK) + os.pathsep +
    str(indian) + os.pathsep +
    env.get("PYTHONPATH", "")
)

result = subprocess.run(
    [
        sys.executable,
        str(indian / "build_indian_testset.py"),
        "--docs_dir", str(indian / "documents"),
        "--clauses_csv", str(indian / "clauses.csv"),
        "--out_dir", str(indian / "built")
    ],
    capture_output=True,
    text=True,
    cwd=str(WORK),
    env=env
)

print("Return code:", result.returncode)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

Return code: 0
# Indian test set build report

2 clauses from 1 documents.

| Category | clauses | model has training data? |
|---|---|---|
| Confidentiality | 1 | no (never seen in training) |
| Indemnification | 1 | no (never seen in training) |

Categories with fewer than 3 clauses (too thin for per-category scores): Confidentiality, Indemnification

CSV encoding read as: utf-8

## Clauses per document

- aera_rfp_consultant_2025.txt: 2

Clauses in never-seen categories: 2 of 2
NOTE: Dispute Resolution is only 'seen' via CUAD's Covenant Not To Sue (no arbitration/forum clauses), so score it separately from the other seen categories.

## Warnings (rows kept, but check them)
- line 3: text ends in dot leaders or a page number -- probably a table-of-contents line



In [17]:
from pathlib import Path

WORK = Path("/kaggle/working/indian_eval_project")

print("Checkpoint files:")
for p in WORK.rglob("ckpt_joint_cls_span_fold0.pt"):
    print(p, "|", round(p.stat().st_size / (1024**2), 2), "MB")

print("\nConfiguration files:")
for p in WORK.rglob("config.json"):
    print(p)

print("\nEvaluation script:")
p = WORK / "indian_set" / "eval_indian.py"
print(p, "| exists:", p.exists())

Checkpoint files:
/kaggle/working/indian_eval_project/ckpt_joint_cls_span_fold0.pt | 567.29 MB

Configuration files:
/kaggle/working/indian_eval_project/partial/config.json

Evaluation script:
/kaggle/working/indian_eval_project/indian_set/eval_indian.py | exists: True


In [18]:
from pathlib import Path
import shutil

WORK = Path("/kaggle/working/indian_eval_project")
RULE_ROOT = Path("/kaggle/input")

# Find both required files in attached Kaggle datasets
engine_files = list(RULE_ROOT.rglob("rules_engine.py"))
rules_files = list(RULE_ROOT.rglob("indian_legal_rules.csv"))

print("Rule engine files:", engine_files)
print("Rules CSV files:", rules_files)

if not engine_files:
    raise FileNotFoundError("rules_engine.py not found in attached inputs.")

if not rules_files:
    raise FileNotFoundError("indian_legal_rules.csv not found in attached inputs.")

# Copy into the project root without modifying the source dataset
shutil.copy2(engine_files[0], WORK / "rules_engine.py")
shutil.copy2(rules_files[0], WORK / "indian_legal_rules.csv")

print("\nCopied successfully:")
print("Engine:", (WORK / "rules_engine.py").exists())
print("Rules:", (WORK / "indian_legal_rules.csv").exists())

Rule engine files: [PosixPath('/kaggle/input/datasets/riyasuryawanshi/rule-engine-v2/rules_engine.py')]
Rules CSV files: [PosixPath('/kaggle/input/datasets/riyasuryawanshi/scan-vs-span/indian_legal_rules.csv'), PosixPath('/kaggle/input/datasets/riyasuryawanshi/cuad-project/indian_legal_rules.csv'), PosixPath('/kaggle/input/datasets/riyasuryawanshi/cuad-project/phase3/indian_legal_rules.csv')]

Copied successfully:
Engine: True
Rules: True


In [19]:
from pathlib import Path
import subprocess
import sys
import os

WORK = Path("/kaggle/working/indian_eval_project")
indian = WORK / "indian_set"

env = os.environ.copy()
env["PYTHONPATH"] = (
    str(WORK) + os.pathsep +
    str(indian) + os.pathsep +
    env.get("PYTHONPATH", "")
)

cmd = [
    sys.executable,
    str(indian / "eval_indian.py"),
    "--run_dir", str(WORK),
    "--fold", "0",
    "--ckpt", str(WORK / "ckpt_joint_cls_span_fold0.pt"),
    "--indian_dir", str(indian / "built"),
    "--mapped_jsonl", str(WORK / "clauses_mapped.jsonl"),
    "--cuad_json", str(WORK / "CUAD_v1.json"),
    "--rules", str(WORK / "indian_legal_rules.csv"),
]

print("Starting evaluation...")
print("Command:", " ".join(cmd))

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    cwd=str(WORK),
    env=env
)

print("\nReturn code:", result.returncode)
print("\nSTDOUT:\n", result.stdout[-12000:])

if result.stderr:
    print("\nSTDERR:\n", result.stderr[-6000:])

Starting evaluation...
Command: /usr/bin/python3 /kaggle/working/indian_eval_project/indian_set/eval_indian.py --run_dir /kaggle/working/indian_eval_project --fold 0 --ckpt /kaggle/working/indian_eval_project/ckpt_joint_cls_span_fold0.pt --indian_dir /kaggle/working/indian_eval_project/indian_set/built --mapped_jsonl /kaggle/working/indian_eval_project/clauses_mapped.jsonl --cuad_json /kaggle/working/indian_eval_project/CUAD_v1.json --rules /kaggle/working/indian_eval_project/indian_legal_rules.csv

Return code: 0

STDOUT:
 2 Indian clauses: 0 in categories the model was trained on, 2 in categories it never saw
# Indian-context evaluation

2 clauses: 0 in categories seen in training, 2 in unseen categories. Small sample: read gaps, not decimals.

## Categories the model never saw: what it predicted instead

| True category | predictions |
|---|---|
| Confidentiality | Termination x1 |
| Indemnification | Intellectual Property x1 |

## Rule-layer findings (see indian_rule_flags.csv)

- 

In [20]:
from pathlib import Path

WORK = Path("/kaggle/working/indian_eval_project")

for p in sorted(WORK.rglob("*")):
    if p.is_file() and (
        "indian" in p.name.lower()
        or "rule" in p.name.lower()
        or "eval" in p.name.lower()
    ):
        print(p.relative_to(WORK), "-", round(p.stat().st_size / 1024, 1), "KB")

__pycache__/rules_engine.cpython-313.pyc - 38.9 KB
indian_legal_rules.csv - 10.9 KB
indian_set/__pycache__/build_indian_testset.cpython-313.pyc - 13.7 KB
indian_set/build_indian_testset.py - 8.6 KB
indian_set/built/indian_contracts.json - 117.6 KB
indian_set/built/indian_eval.json - 0.2 KB
indian_set/built/indian_eval.md - 0.4 KB
indian_set/built/indian_mapped.jsonl - 6.2 KB
indian_set/built/indian_rule_flags.csv - 0.6 KB
indian_set/eval_indian.py - 12.5 KB
rules_engine.py - 28.6 KB


In [21]:
from pathlib import Path
import pandas as pd

WORK = Path("/kaggle/working/indian_eval_project")
built = WORK / "indian_set" / "built"

print("=== INDIAN EVALUATION REPORT ===")
print((built / "indian_eval.md").read_text(encoding="utf-8"))

print("\n=== RULE FLAGS ===")
flags = pd.read_csv(built / "indian_rule_flags.csv")
display(flags)

print("\n=== EVALUATION JSON ===")
print((built / "indian_eval.json").read_text(encoding="utf-8"))

=== INDIAN EVALUATION REPORT ===
# Indian-context evaluation

2 clauses: 0 in categories seen in training, 2 in unseen categories. Small sample: read gaps, not decimals.

## Categories the model never saw: what it predicted instead

| True category | predictions |
|---|---|
| Confidentiality | Termination x1 |
| Indemnification | Intellectual Property x1 |

## Rule-layer findings (see indian_rule_flags.csv)

- ICA_S124_01:possible: 1


=== RULE FLAGS ===


,clause_id,category,rule_id,status,reasons,evidence,note,clause_text
0,indian_aera_rfp_consultant_2025__Indemnificati...,Indemnification,ICA_S124_01,possible,broad scope without a visible cap,all legal expenses,commercial-fairness concern only; Indian law d...,3.6.4\n\nThe Consultant shall hold the Authori...



=== EVALUATION JSON ===
{
 "n_total": 2,
 "n_seen": 0,
 "n_unseen": 2,
 "unseen_category_predictions": {
  "Confidentiality": {
   "Termination": 1
  },
  "Indemnification": {
   "Intellectual Property": 1
  }
 },
 "rule_status_counts": {
  "ICA_S124_01:possible": 1
 }
}


In [22]:
from pathlib import Path
import pandas as pd

WORK = Path("/kaggle/working/indian_eval_project")
INDIAN = WORK / "indian_set"

print("Documents:", len(list((INDIAN / "documents").glob("*.txt"))))
print("Current annotations:")

df = pd.read_csv(INDIAN / "clauses.csv")
display(df)

print("\nAvailable helper files:")
for f in sorted(INDIAN.iterdir()):
    if f.is_file():
        print(f.name)

Documents: 11
Current annotations:


,document_file,category,clause_text,source_note
0,aera_rfp_consultant_2025.txt,Confidentiality,Confidentiality The Consultant and its Key Per...,AERA agreement; Confidentiality section
1,aera_rfp_consultant_2025.txt,Indemnification,3.6.4\n\nThe Consultant shall hold the Authori...,AERA agreement; clause 3.6.4



Available helper files:
aera_rfp_consultant_2025.txt
build_indian_testset.py
clause_browser.csv
clauses.csv
documents_manifest.csv
dot_unified_licence_agreement.txt
eval_indian.py
gem_general_terms_and_conditions.txt
himachal_hemodialysis_ppp_concession_agreement.txt
mea_architect_consultancy_agreement.txt
mea_pmc_services_agreement.txt
meity_msa_cloud_services_template.txt
mha_rfp_project_management_consultants.txt
noida_airport_concession_agreement.txt
prepare_documents.py
smoke_test.py
stqc_non_disclosure_agreement.txt
tender_consultancy_scc_1349.txt


In [23]:
for f in INDIAN.rglob("*clause*browser*.csv"):
    print("Found:", f)
    display(pd.read_csv(f).head(10))

Found: /kaggle/working/indian_eval_project/indian_set/clause_browser.csv


,document_file,"category_hint (keyword only, not a label)",paragraph_no,kind,heading,preview
0,aera_rfp_consultant_2025.txt,Termination,748,heading,"COMMENCEMENT, COMPLETION AND TERMINATION OF AG...","COMMENCEMENT, COMPLETION AND TERMINATION OF AG..."
1,aera_rfp_consultant_2025.txt,Termination,774,heading,Termination of Agreement,Termination of Agreement 2.10.1 By the Authori...
2,aera_rfp_consultant_2025.txt,Termination,791,heading,"Authority, in its sole discretion and for any ...","Authority, in its sole discretion and for any ..."
3,aera_rfp_consultant_2025.txt,Indemnification,821,heading,Indemnity As per GeM GTC.,Indemnity As per GeM GTC. 3.2 Conflict of Inte...
4,aera_rfp_consultant_2025.txt,Confidentiality,685,heading,“Confidential Information” shall have the mean...,“Confidential Information” shall have the mean...
5,aera_rfp_consultant_2025.txt,Confidentiality,852,heading,b) was obtained from a third party with no kno...,b) was obtained from a third party with no kno...
6,aera_rfp_consultant_2025.txt,Dispute Resolution,690,heading,“Dispute” shall have the meaning set forth in ...,“Dispute” shall have the meaning set forth in ...
7,aera_rfp_consultant_2025.txt,Dispute Resolution,796,heading,d) the Authority fails to comply with any fina...,d) the Authority fails to comply with any fina...
8,aera_rfp_consultant_2025.txt,Dispute Resolution,930,heading,Dispute resolution between the buyer and the s...,Dispute resolution between the buyer and the s...
9,aera_rfp_consultant_2025.txt,Governing Law / Jurisdiction,818,heading,Applicable Laws,Applicable Laws The Consultant shall perform t...


In [24]:
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working/indian_eval_project")
INDIAN = WORK / "indian_set"

browser = pd.read_csv(INDIAN / "clause_browser.csv")
print("Total candidate entries:", len(browser))
print("\nEntry types:")
print(browser["kind"].value_counts(dropna=False))

# Keep entries that may contain actual clause text
candidates = browser[
    browser["kind"].astype(str).str.lower().ne("heading")
].copy()

# Remove very short or empty previews
candidates["preview"] = candidates["preview"].fillna("").astype(str)
candidates = candidates[
    candidates["preview"].str.len().between(100, 1500)
]

print("\nCandidate clauses:", len(candidates))
display(
    candidates[
        ["document_file", "category_hint (keyword only, not a label)",
         "paragraph_no", "kind", "preview"]
    ].head(30)
)

Total candidate entries: 471

Entry types:
kind
heading    370
body       101
Name: count, dtype: int64

Candidate clauses: 101


,document_file,"category_hint (keyword only, not a label)",paragraph_no,kind,preview
36,aera_rfp_consultant_2025.txt,Penalties,189,body,AERA reserves the right to extend the time per...
37,aera_rfp_consultant_2025.txt,Penalties,506,body,5. AERA will be entitled to deduct/ recover th...
38,aera_rfp_consultant_2025.txt,Penalties,509,body,"6. Notwithstanding the above, AERA reserves th..."
39,aera_rfp_consultant_2025.txt,Penalties,757,body,Extension of Contract Period: AERA reserves th...
40,aera_rfp_consultant_2025.txt,Non-Solicitation,517,body,"(a) “corrupt practice” means (i) the offering,..."
41,aera_rfp_consultant_2025.txt,Non-Solicitation,838,body,"“corrupt practice” means the offering, giving,..."
42,aera_rfp_consultant_2025.txt,Non-Compete,339,body,Bidder should not have a conflict of interest ...
43,aera_rfp_consultant_2025.txt,Non-Compete,521,body,(e) “restrictive practice” means forming a car...
44,aera_rfp_consultant_2025.txt,Non-Compete,848,body,“restrictive practice” means forming a cartel ...
45,aera_rfp_consultant_2025.txt,Indemnification,873,body,The Consultant shall hold the Authority harmle...


In [25]:
print("Candidate counts by document:")
display(
    candidates.groupby("document_file")
    .size()
    .sort_values(ascending=False)
    .rename("candidate_count")
    .reset_index()
)

print("\nCandidate counts by suggested category:")
display(
    candidates["category_hint (keyword only, not a label)"]
    .value_counts()
    .rename_axis("suggested_category")
    .reset_index(name="count")
)

Candidate counts by document:


,document_file,candidate_count
0,aera_rfp_consultant_2025.txt,18
1,meity_msa_cloud_services_template.txt,18
2,noida_airport_concession_agreement.txt,15
3,himachal_hemodialysis_ppp_concession_agreement...,13
4,dot_unified_licence_agreement.txt,10
5,gem_general_terms_and_conditions.txt,9
6,mha_rfp_project_management_consultants.txt,7
7,stqc_non_disclosure_agreement.txt,6
8,mea_pmc_services_agreement.txt,3
9,mea_architect_consultancy_agreement.txt,2



Candidate counts by suggested category:


,suggested_category,count
0,Penalties,27
1,Force Majeure,26
2,Indemnification,17
3,Confidentiality,16
4,Non-Compete,11
5,Non-Solicitation,4


In [26]:
from pathlib import Path
import pandas as pd
import re

WORK = Path("/kaggle/working/indian_eval_project")
INDIAN = WORK / "indian_set"

browser = pd.read_csv(INDIAN / "clause_browser.csv")
docs_dir = INDIAN / "documents"

rows = []

for _, row in browser.iterrows():
    if str(row.get("kind", "")).lower() != "body":
        continue

    filename = row["document_file"]
    doc_path = docs_dir / filename

    if not doc_path.exists():
        continue

    text = doc_path.read_text(encoding="utf-8", errors="replace")
    paragraphs = re.split(r"\n\s*\n", text)

    paragraph_no = int(row["paragraph_no"])
    if paragraph_no < 0 or paragraph_no >= len(paragraphs):
        continue

    clause_text = paragraphs[paragraph_no].strip()

    if len(clause_text) < 100:
        continue

    rows.append({
        "document_file": filename,
        "paragraph_no": paragraph_no,
        "category_hint": row.get(
            "category_hint (keyword only, not a label)", ""
        ),
        "clause_text": clause_text,
        "verified_category": "",
        "review_notes": ""
    })

review = pd.DataFrame(rows).drop_duplicates(
    subset=["document_file", "paragraph_no"]
)

out_path = INDIAN / "clause_review.csv"
review.to_csv(out_path, index=False)

print("Review rows:", len(review))
print("Saved to:", out_path)
display(review[
    ["document_file", "paragraph_no", "category_hint", "clause_text"]
].head(5))

Review rows: 98
Saved to: /kaggle/working/indian_eval_project/indian_set/clause_review.csv


,document_file,paragraph_no,category_hint,clause_text
0,aera_rfp_consultant_2025.txt,189,Penalties,AERA reserves the right to extend the time per...
1,aera_rfp_consultant_2025.txt,506,Penalties,5. AERA will be entitled to deduct/ recover th...
2,aera_rfp_consultant_2025.txt,509,Penalties,"6. Notwithstanding the above, AERA reserves th..."
3,aera_rfp_consultant_2025.txt,757,Penalties,Extension of Contract Period: AERA reserves th...
4,aera_rfp_consultant_2025.txt,517,Non-Solicitation,"(a) “corrupt practice” means (i) the offering,..."


In [27]:
from pathlib import Path
import pandas as pd
import re

WORK = Path("/kaggle/working/indian_eval_project")
INDIAN = WORK / "indian_set"

review = pd.read_csv(INDIAN / "clause_review.csv")

# Check the first 5 extracted clauses against the source paragraphs
for _, row in review.head(5).iterrows():
    source = (INDIAN / "documents" / row["document_file"]).read_text(
        encoding="utf-8", errors="replace"
    )
    paragraphs = re.split(r"\n\s*\n", source)
    n = int(row["paragraph_no"])

    extracted = row["clause_text"].strip()
    source_paragraph = paragraphs[n].strip() if n < len(paragraphs) else ""

    print("=" * 80)
    print("Document:", row["document_file"])
    print("Paragraph:", n)
    print("Exact match:", extracted == source_paragraph)
    print("Extracted:", extracted[:250])
    print("Source:   ", source_paragraph[:250])

Document: aera_rfp_consultant_2025.txt
Paragraph: 189
Exact match: True
Extracted: AERA reserves the right to extend the time period for completion of tariff determination exercise beyond six months, at its sole discretion, without any financial implication to AERA and without imposition of liquidated damages to the selected consul
Source:    AERA reserves the right to extend the time period for completion of tariff determination exercise beyond six months, at its sole discretion, without any financial implication to AERA and without imposition of liquidated damages to the selected consul
Document: aera_rfp_consultant_2025.txt
Paragraph: 506
Exact match: True
Extracted: 5. AERA will be entitled to deduct/ recover the Liquidated Damages for the delay, unless covered under Force Majeure conditions and/ or for any reasons not attributable to the consultant, @ 0.5% of the contract value of delayed quantity per week or p
Source:    5. AERA will be entitled to deduct/ recover the Liquidated 

In [28]:
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working/indian_eval_project")
INDIAN = WORK / "indian_set"

review = pd.read_csv(INDIAN / "clause_review.csv")

# Prefer a spread across source agreements
review = review.sample(frac=1, random_state=42)
shortlist = (
    review.groupby("document_file", group_keys=False)
    .head(5)
    .head(45)
    .copy()
)

# These are blank intentionally: hints are not ground-truth labels
shortlist["verified_category"] = ""
shortlist["review_notes"] = ""

out = INDIAN / "clause_shortlist.csv"
shortlist.to_csv(out, index=False)

print("Shortlisted clauses:", len(shortlist))
print("Agreements represented:", shortlist["document_file"].nunique())
print("Saved to:", out)

display(shortlist[
    ["document_file", "paragraph_no", "category_hint", "clause_text"]
].head(10))

Shortlisted clauses: 45
Agreements represented: 10
Saved to: /kaggle/working/indian_eval_project/indian_set/clause_shortlist.csv


,document_file,paragraph_no,category_hint,clause_text
62,meity_msa_cloud_services_template.txt,519,Force Majeure,a. “Force Majeure” shall mean any event beyond...
40,himachal_hemodialysis_ppp_concession_agreement...,499,Indemnification,"Shall be liable for and shall indemnify, prote..."
94,stqc_non_disclosure_agreement.txt,19,Confidentiality,(a) The term “Confidential Information” shall ...
18,dot_unified_licence_agreement.txt,435,Penalties,The LICENSEE shall clearly define the scope of...
81,noida_airport_concession_agreement.txt,832,Non-Compete,(6) not less than (4 (four) kiosks or outlets ...
83,noida_airport_concession_agreement.txt,1963,Non-Compete,Parking of vehicles 1k wll be our endeavour to...
64,meity_msa_cloud_services_template.txt,532,Force Majeure,No delay or nonperformance by either party to ...
42,himachal_hemodialysis_ppp_concession_agreement...,1000,Indemnification,The Concessionaire shall provide fair and just...
10,aera_rfp_consultant_2025.txt,877,Indemnification,Accuracy of Documents The Consultant shall be ...
0,aera_rfp_consultant_2025.txt,189,Penalties,AERA reserves the right to extend the time per...


In [29]:
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working/indian_eval_project")
INDIAN = WORK / "indian_set"

review = pd.read_csv(INDIAN / "clause_review.csv")
review["clause_text"] = review["clause_text"].fillna("").astype(str)
review["category_hint"] = review["category_hint"].fillna("").astype(str)

# Rank candidates by likely relevance to contractual provisions.
# This is a screening aid, NOT an automatic ground-truth label.
terms = [
    "shall", "liable", "indemnify", "confidential", "terminate",
    "termination", "dispute", "arbitration", "governing law",
    "intellectual property", "payment", "penalty", "damages",
    "force majeure", "personal data", "privacy", "warranty",
    "assign", "assignment", "limitation of liability"
]

review["relevance_score"] = review["clause_text"].str.lower().apply(
    lambda text: sum(term in text for term in terms)
)

ranked = review.sort_values(
    ["relevance_score"], ascending=False
).drop_duplicates(
    subset=["document_file", "paragraph_no"]
).head(45).copy()

ranked["verified_category"] = ""
ranked["review_notes"] = ""

out = INDIAN / "clause_shortlist_v2.csv"
ranked.to_csv(out, index=False)

print("Candidates for review:", len(ranked))
print("Agreements represented:", ranked["document_file"].nunique())
print("Saved to:", out)

display(ranked[
    ["document_file", "paragraph_no", "category_hint",
     "relevance_score", "clause_text"]
].head(15))

Candidates for review: 45
Agreements represented: 10
Saved to: /kaggle/working/indian_eval_project/indian_set/clause_shortlist_v2.csv


,document_file,paragraph_no,category_hint,relevance_score,clause_text
11,aera_rfp_consultant_2025.txt,769,Force Majeure,5,Force Majeure If at any time during the contin...
80,noida_airport_concession_agreement.txt,1107,Penalties,5,32:7.Penalty for evasion of Fees Inthe event t...
50,mea_pmc_services_agreement.txt,145,Force Majeure,4,10.1 In the event that either party is prevent...
77,noida_airport_concession_agreement.txt,450,Penalties,4,"references to any date, period or Project Mile..."
47,mea_architect_consultancy_agreement.txt,129,Force Majeure,4,In the event that either party is prevented wh...
85,noida_airport_concession_agreement.txt,1382,Indemnification,4,Without limiting the generality ofthe provisio...
10,aera_rfp_consultant_2025.txt,877,Indemnification,4,Accuracy of Documents The Consultant shall be ...
23,dot_unified_licence_agreement.txt,446,Indemnification,3,"30.10 Any dispute, with regard to the provisio..."
2,aera_rfp_consultant_2025.txt,509,Penalties,3,"6. Notwithstanding the above, AERA reserves th..."
1,aera_rfp_consultant_2025.txt,506,Penalties,3,5. AERA will be entitled to deduct/ recover th...


In [30]:
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working/indian_eval_project")
INDIAN = WORK / "indian_set"

ranked = pd.read_csv(INDIAN / "clause_shortlist_v2.csv")

# Preserve existing annotations
existing = pd.read_csv(INDIAN / "clauses.csv")

# Remove candidates already represented by the same source paragraph
# (we retain the existing annotations separately)
review = ranked.copy()
review["verified_category"] = ""
review["include_in_testset"] = ""
review["review_notes"] = ""

# Keep the clause text readable in the review sheet
review["clause_length"] = review["clause_text"].fillna("").str.len()

out = INDIAN / "category_review.csv"
review.to_csv(out, index=False)

print("Review sheet:", out)
print("Candidates:", len(review))
print("Existing annotations preserved:", len(existing))
display(review[
    ["document_file", "paragraph_no", "category_hint",
     "clause_length", "clause_text"]
].head(15))

Review sheet: /kaggle/working/indian_eval_project/indian_set/category_review.csv
Candidates: 45
Existing annotations preserved: 2


,document_file,paragraph_no,category_hint,clause_length,clause_text
0,aera_rfp_consultant_2025.txt,769,Force Majeure,1741,Force Majeure If at any time during the contin...
1,noida_airport_concession_agreement.txt,1107,Penalties,546,32:7.Penalty for evasion of Fees Inthe event t...
2,mea_pmc_services_agreement.txt,145,Force Majeure,794,10.1 In the event that either party is prevent...
3,noida_airport_concession_agreement.txt,450,Penalties,3558,"references to any date, period or Project Mile..."
4,mea_architect_consultancy_agreement.txt,129,Force Majeure,789,In the event that either party is prevented wh...
5,noida_airport_concession_agreement.txt,1382,Indemnification,1069,Without limiting the generality ofthe provisio...
6,aera_rfp_consultant_2025.txt,877,Indemnification,862,Accuracy of Documents The Consultant shall be ...
7,dot_unified_licence_agreement.txt,446,Indemnification,383,"30.10 Any dispute, with regard to the provisio..."
8,aera_rfp_consultant_2025.txt,509,Penalties,242,"6. Notwithstanding the above, AERA reserves th..."
9,aera_rfp_consultant_2025.txt,506,Penalties,459,5. AERA will be entitled to deduct/ recover th...


In [31]:
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working/indian_eval_project")
INDIAN = WORK / "indian_set"

review = pd.read_csv(INDIAN / "category_review.csv")

# Display the first 10 candidates with their full text
for i, row in review.head(10).iterrows():
    print("\n" + "=" * 90)
    print(f"Candidate {i + 1}")
    print("Document:", row["document_file"])
    print("Paragraph:", row["paragraph_no"])
    print("Keyword hint:", row["category_hint"])
    print("Length:", len(str(row["clause_text"])))
    print("\nFULL CLAUSE TEXT:\n")
    print(row["clause_text"])


Candidate 1
Document: aera_rfp_consultant_2025.txt
Paragraph: 769
Keyword hint: Force Majeure
Length: 1741

FULL CLAUSE TEXT:

Force Majeure If at any time during the continuance of the Contract, the performance in whole or in part by either party of any obligation under this Contract shall be prevented or delayed by the reasons of any war, hostility, acts of the public enemy, epidemics, civil commotion, sabotage, fires, floods, explosion, quarantine restrictions, strikes, lockouts or act of God (but not including negligence or wrongdoing, predictable/ seasonal rain) provided notice of happening of such event duly evidenced with documents is given by one party to the other within10 days from the date of occurrence thereof, neither party shall be by reasons of such event, be entitled to terminate the Contract nor shall either party have any claim for damages against the other in respect of such non-performance or the delay in performance, and deliveries under the contract shall be resu

In [32]:
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working/indian_eval_project")
INDIAN = WORK / "indian_set"

review_path = INDIAN / "category_review.csv"
review = pd.read_csv(review_path)

labels = {
    0: ("Force Majeure", "yes", "Force majeure and contract performance"),
    1: ("Penalties", "yes", "Damages for fee evasion"),
    2: ("Force Majeure", "yes", "Force majeure and termination"),
    3: ("Commercial Terms", "no", "Overly broad definitions and interpretation"),
    4: ("Force Majeure", "no", "Duplicate of Candidate 3"),
    5: ("Indemnification", "yes", "Intellectual property infringement indemnity"),
    6: ("Indemnification", "yes", "Indemnity for inaccurate work"),
    7: ("Indemnification", "yes", "Licensor indemnity"),
    8: ("Commercial Terms", "no", "Contract-period extension"),
    9: ("Penalties", "yes", "Liquidated damages for delay"),
}

for idx, (category, include, note) in labels.items():
    review.loc[idx, "verified_category"] = category
    review.loc[idx, "include_in_testset"] = include
    review.loc[idx, "review_notes"] = note

review.to_csv(review_path, index=False)

print("Saved verified labels:", review_path)
display(review.loc[:9, [
    "document_file",
    "category_hint",
    "verified_category",
    "include_in_testset",
    "review_notes"
]])

Saved verified labels: /kaggle/working/indian_eval_project/indian_set/category_review.csv


/tmp/ipykernel_49/3255996447.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Force Majeure' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  review.loc[idx, "verified_category"] = category
/tmp/ipykernel_49/3255996447.py:25: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'yes' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  review.loc[idx, "include_in_testset"] = include
/tmp/ipykernel_49/3255996447.py:26: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Force majeure and contract performance' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  review.loc[idx, "review_notes"] = note


,document_file,category_hint,verified_category,include_in_testset,review_notes
0,aera_rfp_consultant_2025.txt,Force Majeure,Force Majeure,yes,Force majeure and contract performance
1,noida_airport_concession_agreement.txt,Penalties,Penalties,yes,Damages for fee evasion
2,mea_pmc_services_agreement.txt,Force Majeure,Force Majeure,yes,Force majeure and termination
3,noida_airport_concession_agreement.txt,Penalties,Commercial Terms,no,Overly broad definitions and interpretation
4,mea_architect_consultancy_agreement.txt,Force Majeure,Force Majeure,no,Duplicate of Candidate 3
5,noida_airport_concession_agreement.txt,Indemnification,Indemnification,yes,Intellectual property infringement indemnity
6,aera_rfp_consultant_2025.txt,Indemnification,Indemnification,yes,Indemnity for inaccurate work
7,dot_unified_licence_agreement.txt,Indemnification,Indemnification,yes,Licensor indemnity
8,aera_rfp_consultant_2025.txt,Penalties,Commercial Terms,no,Contract-period extension
9,aera_rfp_consultant_2025.txt,Penalties,Penalties,yes,Liquidated damages for delay


In [33]:
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working/indian_eval_project")
INDIAN = WORK / "indian_set"
review_path = INDIAN / "category_review.csv"

review = pd.read_csv(review_path)

# Ensure review fields are text columns
for col in ["verified_category", "include_in_testset", "review_notes"]:
    review[col] = review[col].fillna("").astype(str)

review.to_csv(review_path, index=False)

print("Review file saved:", review_path)
print("\nVerified labels so far:")
print(review["verified_category"].ne("").sum())
print("Marked for inclusion:", (review["include_in_testset"] == "yes").sum())
print("Marked for exclusion:", (review["include_in_testset"] == "no").sum())

display(review.loc[:9, [
    "verified_category", "include_in_testset", "review_notes"
]])

Review file saved: /kaggle/working/indian_eval_project/indian_set/category_review.csv

Verified labels so far:
10
Marked for inclusion: 7
Marked for exclusion: 3


,verified_category,include_in_testset,review_notes
0,Force Majeure,yes,Force majeure and contract performance
1,Penalties,yes,Damages for fee evasion
2,Force Majeure,yes,Force majeure and termination
3,Commercial Terms,no,Overly broad definitions and interpretation
4,Force Majeure,no,Duplicate of Candidate 3
5,Indemnification,yes,Intellectual property infringement indemnity
6,Indemnification,yes,Indemnity for inaccurate work
7,Indemnification,yes,Licensor indemnity
8,Commercial Terms,no,Contract-period extension
9,Penalties,yes,Liquidated damages for delay


**11-20**

In [34]:
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working/indian_eval_project")
INDIAN = WORK / "indian_set"

review = pd.read_csv(INDIAN / "category_review.csv")

for i in range(10, min(20, len(review))):
    row = review.iloc[i]

    print("\n" + "=" * 90)
    print(f"Candidate {i + 1}")
    print("Document:", row["document_file"])
    print("Paragraph:", row["paragraph_no"])
    print("Keyword hint:", row["category_hint"])
    print("Already labelled:", row["verified_category"] or "No")
    print("Length:", len(str(row["clause_text"])))
    print("\nFULL CLAUSE TEXT:\n")
    print(row["clause_text"])


Candidate 11
Document: gem_general_terms_and_conditions.txt
Paragraph: 246
Keyword hint: Force Majeure
Already labelled: nan
Length: 473

FULL CLAUSE TEXT:

Liquidated Damages: If the Seller fails to deliver any or all of the Goods/Services within the original/re-fixed delivery period(s) specified in the contract, the Buyer will be entitled to deduct/recover the Liquidated Damages for the delay, unless covered under Force Majeure conditions aforesaid, @ 0.5% per week or part of the week of delayed period as pre-estimated damages not exceeding 10% of the contract value without any controversy/dispute of any sort whatsoever.

Candidate 12
Document: himachal_hemodialysis_ppp_concession_agreement.txt
Paragraph: 870
Keyword hint: Force Majeure
Already labelled: nan
Length: 340

FULL CLAUSE TEXT:

Save and except as expressly provided in this Article neither party hereto shall be liable in any manner whatsoever to other Party in respect of any loss, damage, cost, expense, claims, demand and

In [35]:
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working/indian_eval_project")
INDIAN = WORK / "indian_set"
path = INDIAN / "category_review.csv"

review = pd.read_csv(path)

# Normalize review columns to avoid pandas dtype warnings
for col in ["verified_category", "include_in_testset", "review_notes"]:
    review[col] = review[col].fillna("").astype(str)

labels = {
    10: ("Penalties", "yes", "Liquidated damages for delayed delivery"),
    11: ("Force Majeure", "yes", "Allocation of losses from force majeure"),
    12: ("Termination", "no", "Contract duration; overlaps with candidate 20"),
    13: ("Commercial Terms", "no", "Contract-period extension"),
    14: ("Confidentiality", "yes", "Restrictions on disclosure of confidential information"),
    15: ("Other", "no", "Table of contents, not a substantive clause"),
    16: ("Indemnification", "yes", "Licensee indemnifies the licensor"),
    17: ("Assignment", "yes", "Transfer and assignment of licence agreement"),
    18: ("Indemnification", "no", "Clause text appears truncated; verify complete source"),
    19: ("Termination", "no", "Contract duration; overlaps with candidate 13"),
}

for idx, (category, include, note) in labels.items():
    review.loc[idx, "verified_category"] = category
    review.loc[idx, "include_in_testset"] = include
    review.loc[idx, "review_notes"] = note

review.to_csv(path, index=False)

print("Saved:", path)
print("Total labelled:", review["verified_category"].ne("").sum())
print("Included:", (review["include_in_testset"] == "yes").sum())
print("Excluded:", (review["include_in_testset"] == "no").sum())

display(review.loc[10:19, [
    "verified_category", "include_in_testset", "review_notes"
]])

Saved: /kaggle/working/indian_eval_project/indian_set/category_review.csv
Total labelled: 20
Included: 12
Excluded: 8


,verified_category,include_in_testset,review_notes
10,Penalties,yes,Liquidated damages for delayed delivery
11,Force Majeure,yes,Allocation of losses from force majeure
12,Termination,no,Contract duration; overlaps with candidate 20
13,Commercial Terms,no,Contract-period extension
14,Confidentiality,yes,Restrictions on disclosure of confidential inf...
15,Other,no,"Table of contents, not a substantive clause"
16,Indemnification,yes,Licensee indemnifies the licensor
17,Assignment,yes,Transfer and assignment of licence agreement
18,Indemnification,no,Clause text appears truncated; verify complete...
19,Termination,no,Contract duration; overlaps with candidate 13


In [36]:
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working/indian_eval_project")
INDIAN = WORK / "indian_set"

review = pd.read_csv(INDIAN / "category_review.csv")

for i in range(20, min(30, len(review))):
    row = review.iloc[i]

    print("\n" + "=" * 90)
    print(f"Candidate {i + 1}")
    print("Document:", row["document_file"])
    print("Paragraph:", row["paragraph_no"])
    print("Keyword hint:", row["category_hint"])
    print("Length:", len(str(row["clause_text"])))
    print("\nFULL CLAUSE TEXT:\n")
    print(row["clause_text"])


Candidate 21
Document: meity_msa_cloud_services_template.txt
Paragraph: 405
Keyword hint: Penalties
Length: 171

FULL CLAUSE TEXT:

g) Payments shall be subject to the application of liquidated damages or SLA penalties and its adjustments/corrections as may be provided for in the Agreement and the SLA.

Candidate 22
Document: stqc_non_disclosure_agreement.txt
Paragraph: 19
Keyword hint: Confidentiality
Length: 1035

FULL CLAUSE TEXT:

(a) The term “Confidential Information” shall include, without limitation, all information and materials, furnished by either Party to the other in connection with Auditee products and services including information transmitted in writing, orally, visually (e.g. video terminal display) or on magnetic media, and including all proprietary information, customer & prospect lists, trade secrets, trade names or proposed trade names, methods and procedures of operation, business or marketing plans, licensed document know-how, ideas, concepts, designs, drawings,

In [38]:
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working/indian_eval_project")
INDIAN = WORK / "indian_set"
path = INDIAN / "category_review.csv"

review = pd.read_csv(path)

for col in ["verified_category", "include_in_testset", "review_notes"]:
    review[col] = review[col].fillna("").astype(str)

labels = {
    20: ("Penalties", "yes", "Liquidated damages and SLA penalties"),
    21: ("Confidentiality", "yes", "Definition of confidential information"),
    22: ("Force Majeure", "yes", "Liability relating to force majeure"),
    23: ("Penalties", "yes", "Damages for delayed conditions precedent"),
    24: ("Indemnification", "yes", "Handling and defence of indemnity claims; review length"),
    25: ("Force Majeure", "yes", "Excused performance and notification"),
    26: ("Commercial Terms", "no", "Duplicates earlier contract-extension subject"),
    27: ("Confidentiality", "yes", "Duty to maintain secrecy"),
    28: ("Penalties", "yes", "Calculation of liquidated damages"),
    29: ("Force Majeure", "no", "Incomplete clause text"),
}

for idx, (category, include, note) in labels.items():
    review.loc[idx, "verified_category"] = category
    review.loc[idx, "include_in_testset"] = include
    review.loc[idx, "review_notes"] = note

review.to_csv(path, index=False)

print("Saved:", path)
print("Total labelled:", review["verified_category"].ne("").sum())
print("Included:", (review["include_in_testset"] == "yes").sum())
print("Excluded:", (review["include_in_testset"] == "no").sum())

display(review.loc[20:29, [
    "verified_category", "include_in_testset", "review_notes"
]])

Saved: /kaggle/working/indian_eval_project/indian_set/category_review.csv
Total labelled: 30
Included: 20
Excluded: 10


,verified_category,include_in_testset,review_notes
20,Penalties,yes,Liquidated damages and SLA penalties
21,Confidentiality,yes,Definition of confidential information
22,Force Majeure,yes,Liability relating to force majeure
23,Penalties,yes,Damages for delayed conditions precedent
24,Indemnification,yes,Handling and defence of indemnity claims; revi...
25,Force Majeure,yes,Excused performance and notification
26,Commercial Terms,no,Duplicates earlier contract-extension subject
27,Confidentiality,yes,Duty to maintain secrecy
28,Penalties,yes,Calculation of liquidated damages
29,Force Majeure,no,Incomplete clause text


In [39]:
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working/indian_eval_project")
INDIAN = WORK / "indian_set"

review = pd.read_csv(INDIAN / "category_review.csv")

for i in range(30, min(40, len(review))):
    row = review.iloc[i]

    print("\n" + "=" * 90)
    print(f"Candidate {i + 1}")
    print("Document:", row["document_file"])
    print("Paragraph:", row["paragraph_no"])
    print("Keyword hint:", row["category_hint"])
    print("Length:", len(str(row["clause_text"])))
    print("\nFULL CLAUSE TEXT:\n")
    print(row["clause_text"])


Candidate 31
Document: dot_unified_licence_agreement.txt
Paragraph: 435
Keyword hint: Penalties
Length: 650

FULL CLAUSE TEXT:

The LICENSEE shall clearly define the scope of Service to the Subscriber(s) at the time of entering into contract with such Subscriber(s). Licensor shall have right to impose suitable penalty, not limited to a financial penalty, apart from any other actions for breach of this condition. The LICENSEE shall commence the Service on commercial basis only after starting subscriber registration in the manner prescribed. Before commencement of Service in an area, the LICENSEE shall notify and publicize the address where any subscriber can register demand /request for Telecom Service. Any change of this address shall be duly notified by the Licensee.

Candidate 32
Document: gem_general_terms_and_conditions.txt
Paragraph: 9
Keyword hint: Penalties
Length: 612

FULL CLAUSE TEXT:

d. SERVICE LEVEL AGREEMENT (SLA)” shall mean the Contractual Commitment that prevails betw

In [40]:
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working/indian_eval_project")
INDIAN = WORK / "indian_set"
path = INDIAN / "category_review.csv"

review = pd.read_csv(path)

for col in ["verified_category", "include_in_testset", "review_notes"]:
    review[col] = review[col].fillna("").astype(str)

labels = {
    30: ("Penalties", "yes", "Penalty for failure to define service scope"),
    31: ("Commercial Terms", "yes", "SLA commitments and remedies"),
    32: ("Indemnification", "yes", "Indemnity for unauthorised use and breach"),
    33: ("Confidentiality", "yes", "Disclosure to professional advisers"),
    34: ("Non-Solicitation", "yes", "Restriction on soliciting employees and consultants"),
    35: ("Confidentiality", "yes", "Restriction on unauthorised disclosure"),
    36: ("Penalties", "yes", "Damages and litigation costs after breach"),
    37: ("Other", "no", "Passenger facilities and service standards"),
    38: ("Confidentiality", "yes", "Definition and protection of confidential information"),
    39: ("Confidentiality", "yes", "Security measures for confidential information"),
}

for idx, (category, include, note) in labels.items():
    review.loc[idx, "verified_category"] = category
    review.loc[idx, "include_in_testset"] = include
    review.loc[idx, "review_notes"] = note

review.to_csv(path, index=False)

print("Saved:", path)
print("Total labelled:", review["verified_category"].ne("").sum())
print("Included:", (review["include_in_testset"] == "yes").sum())
print("Excluded:", (review["include_in_testset"] == "no").sum())

display(review.loc[30:39, [
    "verified_category", "include_in_testset", "review_notes"
]])

Saved: /kaggle/working/indian_eval_project/indian_set/category_review.csv
Total labelled: 40
Included: 29
Excluded: 11


,verified_category,include_in_testset,review_notes
30,Penalties,yes,Penalty for failure to define service scope
31,Commercial Terms,yes,SLA commitments and remedies
32,Indemnification,yes,Indemnity for unauthorised use and breach
33,Confidentiality,yes,Disclosure to professional advisers
34,Non-Solicitation,yes,Restriction on soliciting employees and consul...
35,Confidentiality,yes,Restriction on unauthorised disclosure
36,Penalties,yes,Damages and litigation costs after breach
37,Other,no,Passenger facilities and service standards
38,Confidentiality,yes,Definition and protection of confidential info...
39,Confidentiality,yes,Security measures for confidential information


In [41]:
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working/indian_eval_project")
INDIAN = WORK / "indian_set"

review = pd.read_csv(INDIAN / "category_review.csv")

for i in range(40, len(review)):
    row = review.iloc[i]

    print("\n" + "=" * 90)
    print(f"Candidate {i + 1}")
    print("Document:", row["document_file"])
    print("Paragraph:", row["paragraph_no"])
    print("Keyword hint:", row["category_hint"])
    print("Length:", len(str(row["clause_text"])))
    print("\nFULL CLAUSE TEXT:\n")
    print(row["clause_text"])


Candidate 41
Document: meity_msa_cloud_services_template.txt
Paragraph: 594
Keyword hint: Confidentiality
Length: 209

FULL CLAUSE TEXT:

The Managed Service Provider shall keep confidential all the details and information with regard to the Project, including systems, facilities, operations, management and maintenance of the systems/facilities.

Candidate 42
Document: mha_rfp_project_management_consultants.txt
Paragraph: 422
Keyword hint: Penalties
Length: 700

FULL CLAUSE TEXT:

The Bidder shall ensure that none of the Key Personnel (refer Clause 10.2 of Section 5 of the RFP) proposed, exit from the project during first 18 months of the project. The change of Key Personnel will be accepted only in case of person leaving the organization or medical exigency. It is important that the Bidder submits profiles of only those resources that are available for the project. In case of any change of the Key Personnel during the first 18 months of the project, it would attract a penalty as per 

In [42]:
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working/indian_eval_project")
INDIAN = WORK / "indian_set"
path = INDIAN / "category_review.csv"

review = pd.read_csv(path)

for col in ["verified_category", "include_in_testset", "review_notes"]:
    review[col] = review[col].fillna("").astype(str)

labels = {
    40: ("Confidentiality", "yes", "Protection of project information"),
    41: ("Penalties", "yes", "Penalty for changing key personnel"),
    42: ("Confidentiality", "yes", "Reporting unauthorised access attempts"),
    43: ("Indemnification", "yes", "General indemnity obligations of both parties"),
    44: ("Confidentiality", "yes", "Reporting information-security breaches"),
}

for idx, (category, include, note) in labels.items():
    review.loc[idx, "verified_category"] = category
    review.loc[idx, "include_in_testset"] = include
    review.loc[idx, "review_notes"] = note

review.to_csv(path, index=False)

print("Saved:", path)
print("Total labelled:", review["verified_category"].ne("").sum())
print("Included:", (review["include_in_testset"] == "yes").sum())
print("Excluded:", (review["include_in_testset"] == "no").sum())

print("\nCategory counts among included candidates:")
display(
    review[review["include_in_testset"] == "yes"]
    ["verified_category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="count")
)

Saved: /kaggle/working/indian_eval_project/indian_set/category_review.csv
Total labelled: 45
Included: 34
Excluded: 11

Category counts among included candidates:


,category,count
0,Confidentiality,10
1,Penalties,9
2,Indemnification,7
3,Force Majeure,5
4,Assignment,1
5,Commercial Terms,1
6,Non-Solicitation,1


In [43]:
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working/indian_eval_project")
INDIAN = WORK / "indian_set"

review = pd.read_csv(INDIAN / "category_review.csv").fillna("")
original = pd.read_csv(INDIAN / "clauses.csv").fillna("")

# Select only manually reviewed candidates marked for inclusion
accepted = review[
    review["include_in_testset"].astype(str).str.lower().eq("yes")
].copy()

accepted = accepted.rename(columns={
    "verified_category": "category",
    "review_notes": "source_note"
})

accepted = accepted[
    ["document_file", "category", "clause_text", "source_note"]
]

# Combine with the two original annotations
final_df = pd.concat(
    [
        original[["document_file", "category", "clause_text", "source_note"]],
        accepted
    ],
    ignore_index=True
)

# Flag exact duplicate clauses for review
final_df["normalized_text"] = (
    final_df["clause_text"]
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print("Original annotations:", len(original))
print("Accepted candidates:", len(accepted))
print("Combined rows:", len(final_df))
print("Exact duplicate texts:", final_df["normalized_text"].duplicated().sum())

print("\nCategory distribution:")
display(
    final_df["category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="count")
)

out = INDIAN / "clauses_expanded_draft.csv"
final_df.drop(columns=["normalized_text"]).to_csv(out, index=False)

print("\nSaved draft:", out)

Original annotations: 2
Accepted candidates: 34
Combined rows: 36
Exact duplicate texts: 0

Category distribution:


,category,count
0,Confidentiality,11
1,Penalties,9
2,Indemnification,8
3,Force Majeure,5
4,Assignment,1
5,Commercial Terms,1
6,Non-Solicitation,1



Saved draft: /kaggle/working/indian_eval_project/indian_set/clauses_expanded_draft.csv


In [44]:
import pandas as pd
from pathlib import Path
from difflib import SequenceMatcher

WORK = Path("/kaggle/working/indian_eval_project")
INDIAN = WORK / "indian_set"

df = pd.read_csv(INDIAN / "clauses_expanded_draft.csv").fillna("")

# 1. Check category compatibility
allowed = {
    "Assignment", "Commercial Terms", "Dispute Resolution",
    "Governing Law / Jurisdiction", "Intellectual Property",
    "Liability", "Non-Compete", "Non-Solicitation", "Other",
    "Penalties", "Termination", "Warranty", "Confidentiality",
    "Force Majeure", "Indemnification", "Modification",
    "Payment", "Privacy / Data Protection"
}

print("Unsupported categories:")
print(sorted(set(df["category"]) - allowed))

# 2. Find highly similar clauses (manual review required)
texts = (
    df["clause_text"].astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .tolist()
)

pairs = []
for i in range(len(texts)):
    for j in range(i + 1, len(texts)):
        score = SequenceMatcher(None, texts[i], texts[j]).ratio()
        if score >= 0.75:
            pairs.append({
                "row_a": i,
                "row_b": j,
                "similarity": round(score, 3),
                "category_a": df.iloc[i]["category"],
                "category_b": df.iloc[j]["category"],
                "document_a": df.iloc[i]["document_file"],
                "document_b": df.iloc[j]["document_file"]
            })

similar = pd.DataFrame(pairs)
print("\nNear-duplicate pairs (similarity >= 0.75):", len(similar))

if not similar.empty:
    display(similar.sort_values("similarity", ascending=False))
else:
    print("No pairs crossed the similarity threshold.")

Unsupported categories:
[]

Near-duplicate pairs (similarity >= 0.75): 0
No pairs crossed the similarity threshold.


In [45]:
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working/indian_eval_project")
INDIAN = WORK / "indian_set"

df = pd.read_csv(INDIAN / "clauses_expanded_draft.csv").fillna("")

for idx in [1, 24]:
    row = df.iloc[idx]

    print("\n" + "=" * 80)
    print("Row:", idx)
    print("Document:", row["document_file"])
    print("Category:", row["category"])
    print("Source note:", row["source_note"])
    print("Length:", len(row["clause_text"]))
    print("\nClause text:\n")
    print(row["clause_text"])


Row: 1
Document: aera_rfp_consultant_2025.txt
Category: Indemnification
Source note: AERA agreement; clause 3.6.4
Length: 3046

Clause text:

3.6.4

The Consultant shall hold the Authority harmless and indemnified for any losses, claims, damages, expenses (including all legal expenses), awards, penalties or injuries (collectively referred to as “Claims”) which may arise from or due to any unauthorized use of such Consultancy Documents, or due to any breach or failure on part of the Consultant (or any Third Party appointed by the Consultant) to perform any of its duties or obligations in relation to securing the aforementioned rights of the Authority.

3.7

Materials furnished by the Authority Materials made available to the Consultant by the Authority shall be the property of the Authority and shall be marked accordingly. Upon termination or expiration of this Agreement, the Consultant shall furnish forthwith to the Authority, an inventory of such materials and shall dispose of such m

In [46]:
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working/indian_eval_project")
INDIAN = WORK / "indian_set"

draft_path = INDIAN / "clauses_expanded_draft.csv"
df = pd.read_csv(draft_path).fillna("")

# Remove the duplicate shortlist entry at row index 24.
# Keep the original annotation at row index 1.
clean = df.drop(index=24).reset_index(drop=True)

out = INDIAN / "clauses_expanded_clean.csv"
clean.to_csv(out, index=False)

print("Rows before:", len(df))
print("Rows after:", len(clean))
print("Saved:", out)

print("\nCategory distribution:")
display(
    clean["category"].value_counts()
    .rename_axis("category")
    .reset_index(name="count")
)

Rows before: 36
Rows after: 35
Saved: /kaggle/working/indian_eval_project/indian_set/clauses_expanded_clean.csv

Category distribution:


,category,count
0,Confidentiality,11
1,Penalties,9
2,Indemnification,7
3,Force Majeure,5
4,Assignment,1
5,Commercial Terms,1
6,Non-Solicitation,1


In [47]:
import sys
import subprocess
import os
from pathlib import Path

WORK = Path("/kaggle/working/indian_eval_project")
INDIAN = WORK / "indian_set"

env = os.environ.copy()
env["PYTHONPATH"] = (
    str(WORK) + os.pathsep
    + str(INDIAN) + os.pathsep
    + env.get("PYTHONPATH", "")
)

result = subprocess.run(
    [sys.executable, str(INDIAN / "build_indian_testset.py"), "--help"],
    capture_output=True,
    text=True,
    env=env
)

print(result.stdout)
if result.stderr:
    print("STDERR:\n", result.stderr)

print("\nClean CSV columns:")
import pandas as pd
clean = pd.read_csv(INDIAN / "clauses_expanded_clean.csv")
print(clean.columns.tolist())
print("Rows:", len(clean))

usage: build_indian_testset.py [-h] [--docs_dir DOCS_DIR]
                               [--clauses_csv CLAUSES_CSV] [--out_dir OUT_DIR]
                               [--make_template DIR] [--list_categories]

build_indian_testset.py -- turn your hand-collected Indian clauses into a test set
the Phase 3 code can evaluate.

YOU PROVIDE
    indian_set/documents/*.txt   full plain text of each Indian agreement / template
    indian_set/clauses.csv       one row per clause you picked
        document_file   file name inside documents/ (e.g. nda_sample_1.txt)
        category        one of the 18 ClauseXplain categories (exact spelling, see --list_categories)
        clause_text     the clause, COPIED from that document (whitespace differences are fine)
        source_note     where the document came from (public template, tender, ...)

THIS SCRIPT
    finds each clause inside its document (so we get exact character offsets, which
    the span evaluation needs), and writes
        built/in

In [48]:
import os
import sys
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/indian_eval_project")
INDIAN = WORK / "indian_set"

env = os.environ.copy()
env["PYTHONPATH"] = (
    str(WORK) + os.pathsep
    + str(INDIAN) + os.pathsep
    + env.get("PYTHONPATH", "")
)

cmd = [
    sys.executable,
    str(INDIAN / "build_indian_testset.py"),
    "--docs_dir", str(INDIAN / "documents"),
    "--clauses_csv", str(INDIAN / "clauses_expanded_clean.csv"),
    "--out_dir", str(INDIAN / "built_expanded")
]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    env=env
)

print(result.stdout)
if result.stderr:
    print("STDERR:\n", result.stderr)

print("Return code:", result.returncode)

# Indian test set build report

32 clauses from 9 documents.

| Category | clauses | model has training data? |
|---|---|---|
| Assignment | 1 | yes |
| Commercial Terms | 1 | yes |
| Non-Solicitation | 1 | yes |
| Penalties | 9 | yes |
| Confidentiality | 9 | no (never seen in training) |
| Force Majeure | 5 | no (never seen in training) |
| Indemnification | 6 | no (never seen in training) |

Categories with fewer than 3 clauses (too thin for per-category scores): Assignment, Commercial Terms, Non-Solicitation

CSV encoding read as: utf-8

## Clauses per document

- noida_airport_concession_agreement.txt: 7  <-- more than 6; one document may dominate
- dot_unified_licence_agreement.txt: 6
- aera_rfp_consultant_2025.txt: 4
- meity_msa_cloud_services_template.txt: 4
- stqc_non_disclosure_agreement.txt: 4
- mha_rfp_project_management_consultants.txt: 3
- gem_general_terms_and_conditions.txt: 2
- mea_pmc_services_agreement.txt: 1
- himachal_hemodialysis_ppp_concession_agreement.txt: 1

C

In [49]:
from pathlib import Path
import pandas as pd

WORK = Path("/kaggle/working/indian_eval_project")
INDIAN = WORK / "indian_set"

report_path = INDIAN / "built_expanded" / "build_report.md"
csv_path = INDIAN / "clauses_expanded_clean.csv"

print(report_path.read_text(encoding="utf-8"))

df = pd.read_csv(csv_path).fillna("")

print("\nRows mentioned in the builder warnings:")
display(df.loc[[12, 25], [
    "document_file", "category", "source_note", "clause_text"
]])

# Indian test set build report

32 clauses from 9 documents.

| Category | clauses | model has training data? |
|---|---|---|
| Assignment | 1 | yes |
| Commercial Terms | 1 | yes |
| Non-Solicitation | 1 | yes |
| Penalties | 9 | yes |
| Confidentiality | 9 | no (never seen in training) |
| Force Majeure | 5 | no (never seen in training) |
| Indemnification | 6 | no (never seen in training) |

Categories with fewer than 3 clauses (too thin for per-category scores): Assignment, Commercial Terms, Non-Solicitation

CSV encoding read as: utf-8

## Clauses per document

- noida_airport_concession_agreement.txt: 7  <-- more than 6; one document may dominate
- dot_unified_licence_agreement.txt: 6
- aera_rfp_consultant_2025.txt: 4
- meity_msa_cloud_services_template.txt: 4
- stqc_non_disclosure_agreement.txt: 4
- mha_rfp_project_management_consultants.txt: 3
- gem_general_terms_and_conditions.txt: 2
- mea_pmc_services_agreement.txt: 1
- himachal_hemodialysis_ppp_concession_agreement.txt: 1

C

,document_file,category,source_note,clause_text
12,dot_unified_licence_agreement.txt,Indemnification,Licensee indemnifies the licensor,6.2 The Licensor shall have the right to direc...
25,stqc_non_disclosure_agreement.txt,Non-Solicitation,Restriction on soliciting employees and consul...,18. Non-solicitation. During the term of this ...


In [50]:
import pandas as pd
from pathlib import Path

WORK = Path("/kaggle/working/indian_eval_project")
INDIAN = WORK / "indian_set"

df = pd.read_csv(INDIAN / "clauses_expanded_clean.csv").fillna("")

for skipped_idx in [12, 25]:
    skipped = df.iloc[skipped_idx]
    print(f"\n{'='*80}")
    print(f"SKIPPED ROW — CSV index {skipped_idx}")
    print("Document:", skipped["document_file"])
    print("Category:", skipped["category"])
    print("Source note:", skipped["source_note"])
    print("Clause:", skipped["clause_text"][:1200])

    same_category = df[
        (df["category"] == skipped["category"]) &
        (df.index < skipped_idx)
    ]

    print("\nEarlier clauses with the same category:")
    for idx, row in same_category.iterrows():
        print(f"\nCSV index {idx} | {row['document_file']}")
        print("Source note:", row["source_note"])
        print("Clause:", row["clause_text"][:1200])


SKIPPED ROW — CSV index 12
Document: dot_unified_licence_agreement.txt
Category: Indemnification
Source note: Licensee indemnifies the licensor
Clause: 6.2 The Licensor shall have the right to direct the Licensee to warn, penalize or terminate the services of the franchisee or agent or distributor servant, after considering any report of conduct or antecedents detrimental to the security of the nation. The decision of the Licensor in this regard shall be final and binding and in any case the Licensee shall bear all liabilities in the matter and keep the Licensor indemnified for all claims, cost, charges or damages in this respect.

Earlier clauses with the same category:

CSV index 1 | aera_rfp_consultant_2025.txt
Source note: AERA agreement; clause 3.6.4
Clause: 3.6.4

The Consultant shall hold the Authority harmless and indemnified for any losses, claims, damages, expenses (including all legal expenses), awards, penalties or injuries (collectively referred to as “Claims”) which may 

In [51]:
from pathlib import Path

INDIAN = Path("/kaggle/working/indian_eval_project/indian_set")
builder = INDIAN / "build_indian_testset.py"

lines = builder.read_text(encoding="utf-8").splitlines()

for i, line in enumerate(lines):
    if "DUPLICATE/overlapping" in line:
        start = max(0, i - 12)
        end = min(len(lines), i + 8)
        print(f"\n--- Lines {start + 1}–{end} ---")
        for j in range(start, end):
            print(f"{j + 1}: {lines[j]}")


--- Lines 107–126 ---
107:             if count > 1:
108:                 multi.append(f"line {n}: text occurs {count} times in {fn}; first occurrence used")
109:             s, e = loc
110:             if len(txt.split()) < 12:
111:                 notes.append(f"line {n}: only {len(txt.split())} words -- looks like a heading, not a clause (check it is not a table-of-contents line)")
112:             if re.search(r"\.{4,}|\s\d{1,3}\s*$", docs[fn][s:e]):
113:                 notes.append(f"line {n}: text ends in dot leaders or a page number -- probably a table-of-contents line")
114:             clash = [(o_fn, o_n, o_cat) for (o_fn, o_s, o_e, o_n, o_cat) in seen_spans
115:                      if o_fn == fn and s < o_e and o_s < e]
116:             if clash:
117:                 same = [c for c in clash if c[2] == cat]
118:                 if same:
119:                     problems.append(f"line {n}: DUPLICATE/overlapping row for the same category as line {same[0][1]}; skipped so it 

In [52]:
from pathlib import Path

INDIAN = Path("/kaggle/working/indian_eval_project/indian_set")
builder = INDIAN / "build_indian_testset.py"

lines = builder.read_text(encoding="utf-8").splitlines()

for start, end in [(65, 110), (114, 130)]:
    print(f"\n--- Lines {start + 1}–{end} ---")
    for i in range(start, min(end, len(lines))):
        print(f"{i + 1}: {lines[i]}")


--- Lines 66–110 ---
66:     ap.add_argument("--out_dir", default="indian_set/built")
67:     ap.add_argument("--make_template", metavar="DIR", help="create DIR/documents and DIR/clauses.csv")
68:     ap.add_argument("--list_categories", action="store_true")
69:     a = ap.parse_args(argv)
70: 
71:     if a.list_categories:
72:         print("\n".join(CATEGORIES)); return
73:     if a.make_template:
74:         d = Path(a.make_template); (d / "documents").mkdir(parents=True, exist_ok=True)
75:         with open(d / "clauses.csv", "w", newline="", encoding="utf-8") as f:
76:             w = csv.writer(f)
77:             w.writerow(["document_file", "category", "clause_text", "source_note"])
78:             w.writerow(["example_nda.txt", "Termination", "Either party may terminate this Agreement by giving thirty (30) days written notice.", "public template (example row, delete me)"])
79:         print(f"created {d}/documents and {d}/clauses.csv; put plain-text agreements in documents/");

In [53]:
from pathlib import Path
import pandas as pd
import json

WORK = Path("/kaggle/working/indian_eval_project")
INDIAN = WORK / "indian_set"

mapped_path = INDIAN / "built_expanded" / "indian_mapped.jsonl"

mapped = [
    json.loads(line)
    for line in mapped_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

for filename in [
    "dot_unified_licence_agreement.txt",
    "stqc_non_disclosure_agreement.txt",
]:
    print(f"\n--- {filename} ---")
    for row in mapped:
        if row["contract_id"] == Path(filename).stem:
            print(
                row["clausexplain_category"],
                "start:", row["char_start"],
                "end:", row["char_end"],
                "length:", len(row["clause_text"]),
                "|", row["source_note"]
            )


--- dot_unified_licence_agreement.txt ---

--- stqc_non_disclosure_agreement.txt ---


In [54]:
from pathlib import Path

INDIAN = Path("/kaggle/working/indian_eval_project/indian_set")

report = INDIAN / "built_expanded" / "build_report.md"
mapped = INDIAN / "built_expanded" / "indian_mapped.jsonl"

print("BUILD REPORT")
print(report.read_text(encoding="utf-8"))

print("\nMAPPED FILE EXISTS:", mapped.exists())
if mapped.exists():
    lines = [
        line for line in mapped.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    print("Mapped clause count:", len(lines))
    print("First mapped record:")
    print(lines[0] if lines else "No records")

BUILD REPORT
# Indian test set build report

32 clauses from 9 documents.

| Category | clauses | model has training data? |
|---|---|---|
| Assignment | 1 | yes |
| Commercial Terms | 1 | yes |
| Non-Solicitation | 1 | yes |
| Penalties | 9 | yes |
| Confidentiality | 9 | no (never seen in training) |
| Force Majeure | 5 | no (never seen in training) |
| Indemnification | 6 | no (never seen in training) |

Categories with fewer than 3 clauses (too thin for per-category scores): Assignment, Commercial Terms, Non-Solicitation

CSV encoding read as: utf-8

## Clauses per document

- noida_airport_concession_agreement.txt: 7  <-- more than 6; one document may dominate
- dot_unified_licence_agreement.txt: 6
- aera_rfp_consultant_2025.txt: 4
- meity_msa_cloud_services_template.txt: 4
- stqc_non_disclosure_agreement.txt: 4
- mha_rfp_project_management_consultants.txt: 3
- gem_general_terms_and_conditions.txt: 2
- mea_pmc_services_agreement.txt: 1
- himachal_hemodialysis_ppp_concession_agreem

In [55]:
from pathlib import Path
import json

INDIAN = Path("/kaggle/working/indian_eval_project/indian_set")
mapped_path = INDIAN / "built_expanded" / "indian_mapped.jsonl"

mapped = [
    json.loads(line)
    for line in mapped_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

for row in mapped:
    if any(term in row["contract_id"].lower() for term in ["dot", "stqc"]):
        print(
            row["contract_id"],
            "|", row["clausexplain_category"],
            "|", row["char_start"], row["char_end"],
            "|", row["source_note"]
        )

indian_dot_unified_licence_agreement | Indemnification | 72476 72859 | Licensor indemnity
indian_dot_unified_licence_agreement | Indemnification | 23566 24052 | Licensee indemnifies the licensor
indian_dot_unified_licence_agreement | Assignment | 24227 25691 | Transfer and assignment of licence agreement
indian_stqc_non_disclosure_agreement | Confidentiality | 2742 3777 | Definition of confidential information
indian_dot_unified_licence_agreement | Confidentiality | 83925 84150 | Duty to maintain secrecy
indian_dot_unified_licence_agreement | Penalties | 241861 242081 | Calculation of liquidated damages
indian_dot_unified_licence_agreement | Penalties | 68604 69254 | Penalty for failure to define service scope
indian_stqc_non_disclosure_agreement | Non-Solicitation | 12338 12770 | Restriction on soliciting employees and consultants
indian_stqc_non_disclosure_agreement | Confidentiality | 9537 9695 | Restriction on unauthorised disclosure
indian_stqc_non_disclosure_agreement | Penalties

In [56]:
from collections import Counter

print("Mapped clauses:", len(mapped))

print("\nCategory counts:")
for category, count in sorted(
    Counter(row["clausexplain_category"] for row in mapped).items()
):
    print(f"{category}: {count}")

print("\nTarget clauses:")
for row in mapped:
    if (
        row["source_note"] in [
            "Licensee indemnifies the licensor",
            "Restriction on soliciting employees and consultants",
        ]
    ):
        print(
            row["clausexplain_category"],
            "|", row["source_note"],
            "|", len(row["clause_text"]), "characters"
        )

Mapped clauses: 32

Category counts:
Assignment: 1
Commercial Terms: 1
Confidentiality: 9
Force Majeure: 5
Indemnification: 6
Non-Solicitation: 1
Penalties: 9

Target clauses:
Indemnification | Licensee indemnifies the licensor | 486 characters
Non-Solicitation | Restriction on soliciting employees and consultants | 432 characters


In [57]:
import os
import sys
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/indian_eval_project")
INDIAN = WORK / "indian_set"

env = os.environ.copy()
env["PYTHONPATH"] = (
    str(WORK) + os.pathsep +
    str(INDIAN) + os.pathsep +
    env.get("PYTHONPATH", "")
)

cmd = [
    sys.executable,
    str(INDIAN / "eval_indian.py"),
    "--run_dir", str(WORK),
    "--fold", "0",
    "--ckpt", str(WORK / "ckpt_joint_cls_span_fold0.pt"),
    "--indian_dir", str(INDIAN / "built_expanded"),
    "--mapped_jsonl", str(WORK / "clauses_mapped.jsonl"),
    "--cuad_json", str(WORK / "CUAD_v1.json"),
    "--rules", str(WORK / "indian_legal_rules.csv"),
]

result = subprocess.run(
    cmd,
    env=env,
    capture_output=True,
    text=True,
    cwd=str(WORK)
)

print("Return code:", result.returncode)
print("\nSTDOUT:\n", result.stdout[-15000:])
print("\nSTDERR:\n", result.stderr[-5000:])

Return code: 0

STDOUT:
 32 Indian clauses: 12 in categories the model was trained on, 20 in categories it never saw
# Indian-context evaluation

32 clauses: 12 in categories seen in training, 20 in unseen categories. Small sample: read gaps, not decimals.

## Category classification (seen categories only)

| Model | n | accuracy | 95% interval | macro-F1 |
|---|---|---|---|---|
| ClauseXplain checkpoint | 12 | 0.500 | 0.254 to 0.746 | 0.436 |
| majority (trained on CUAD fold 0) | 12 | 0.000 |  | 0.000 |
| tfidf_logreg (trained on CUAD fold 0) | 12 | 0.417 | 0.193 to 0.680 | 0.422 |

## Span localisation (windowed)

| Method | n | token-F1 | exact match | IoU |
|---|---|---|---|---|
| span head (best-match gold) | 12 | 0.583 | 0.000 | 0.502 |
| span head (own label) | 12 | 0.583 | 0.000 | 0.502 |
| random span | 12 | 0.177 | 0.000 | 0.117 |
| whole_window_baseline | 12 | 0.325 | 0.000 | 0.204 |
| lexical_centroid_baseline | 12 | 0.484 | 0.000 | 0.407 |

## Categories the model never sa

In [58]:
from pathlib import Path

WORK = Path("/kaggle/working/indian_eval_project")
INDIAN = WORK / "indian_set"
OUT = INDIAN / "built_expanded"

print("Evaluation output files:")
for path in sorted(OUT.iterdir()):
    print(path.name, f"({path.stat().st_size:,} bytes)")

Evaluation output files:
build_report.md (1,685 bytes)
indian_contracts.json (1,790,787 bytes)
indian_eval.json (7,620 bytes)
indian_eval.md (1,316 bytes)
indian_mapped.jsonl (39,383 bytes)
indian_rule_flags.csv (3,915 bytes)


In [69]:
import json
from pathlib import Path

WORK = Path("/kaggle/working/indian_eval_project")
eval_path = WORK / "indian_set" / "built_expanded" / "indian_eval.json"

with open(eval_path, "r", encoding="utf-8") as f:
    results = json.load(f)

print("Top-level keys:")
print(list(results.keys()))

print("\nEvaluation summary:")
for key, value in results.items():
    if isinstance(value, (str, int, float, bool)) or value is None:
        print(f"{key}: {value}")
    elif isinstance(value, dict):
        print(f"{key}: dictionary with keys {list(value.keys())[:15]}")
    elif isinstance(value, list):
        print(f"{key}: list with {len(value)} items")
        if value and isinstance(value[0], dict):
            print("  First item's keys:", list(value[0].keys()))
            print("  First item:", json.dumps(value[0], indent=2)[:1800])

Top-level keys:
['n_total', 'n_seen', 'n_unseen', 'cls', 'cls_acc_ci95', 'span', 'span_own_label', 'span_random_baseline', 'baselines_cls', 'tfidf_acc_ci95', 'baselines_span', 'unseen_category_predictions', 'rule_status_counts']

Evaluation summary:
n_total: 33
n_seen: 12
n_unseen: 21
cls: dictionary with keys ['n', 'accuracy', 'micro_f1', 'macro_f1', 'macro_f1_n_classes', 'per_class']
cls_acc_ci95: list with 2 items
span: dictionary with keys ['n', 'token_f1', 'exact_match', 'iou', 'n_truncated_no_gold']
span_own_label: dictionary with keys ['n', 'token_f1', 'exact_match', 'iou']
span_random_baseline: dictionary with keys ['n', 'token_f1', 'exact_match', 'iou']
baselines_cls: dictionary with keys ['majority', 'tfidf_logreg']
tfidf_acc_ci95: list with 2 items
baselines_span: dictionary with keys ['whole_window_baseline', 'lexical_centroid_baseline']
unseen_category_predictions: dictionary with keys ['Confidentiality', 'Indemnification', 'Force Majeure']
rule_status_counts: dictionary w

In [70]:
import json
from pathlib import Path
import pandas as pd

WORK = Path("/kaggle/working/indian_eval_project")
eval_path = WORK / "indian_set" / "built_expanded" / "indian_eval.json"

with open(eval_path, "r", encoding="utf-8") as f:
    results = json.load(f)

print("PER-CLASS CLASSIFICATION METRICS")
display(pd.DataFrame(results["cls"]["per_class"]).T)

print("\nSPAN METRICS")
print(json.dumps(results["span"], indent=2))

print("\nBASELINE CLASSIFICATION METRICS")
print(json.dumps(results["baselines_cls"], indent=2))

PER-CLASS CLASSIFICATION METRICS


,precision,recall,f1,support
Assignment,1.0,1.000000,1.000000,1.0
Commercial Terms,0.0,0.000000,0.000000,1.0
Dispute Resolution,NaN,NaN,NaN,0.0
Governing Law / Jurisdiction,NaN,NaN,NaN,0.0
Intellectual Property,NaN,NaN,NaN,0.0
Liability,NaN,NaN,NaN,0.0
Non-Compete,NaN,NaN,NaN,0.0
Non-Solicitation,1.0,1.000000,1.000000,1.0
Other,NaN,NaN,NaN,0.0
Penalties,1.0,0.444444,0.615385,9.0



SPAN METRICS
{
  "n": 12,
  "token_f1": 0.5831019685273903,
  "exact_match": 0.0,
  "iou": 0.5019278007430243,
  "n_truncated_no_gold": 0
}

BASELINE CLASSIFICATION METRICS
{
  "majority": {
    "n": 12,
    "accuracy": 0.0,
    "micro_f1": 0.0,
    "macro_f1": 0.0,
    "macro_f1_n_classes": 5,
    "per_class": {
      "Assignment": {
        "precision": 0.0,
        "recall": 0.0,
        "f1": 0.0,
        "support": 1
      },
      "Commercial Terms": {
        "precision": 0.0,
        "recall": 0.0,
        "f1": 0.0,
        "support": 1
      },
      "Dispute Resolution": {
        "precision": NaN,
        "recall": NaN,
        "f1": NaN,
        "support": 0
      },
      "Governing Law / Jurisdiction": {
        "precision": NaN,
        "recall": NaN,
        "f1": NaN,
        "support": 0
      },
      "Intellectual Property": {
        "precision": NaN,
        "recall": NaN,
        "f1": NaN,
        "support": 0
      },
      "Liability": {
        "precision":